# 04 - Validación cruzada

En este notebook se aplica validación cruzada para evaluar la estabilidad de los modelos de regresión del proyecto.

El objetivo es cubrir el requisito de Nivel Medio:

- Uso de técnicas de Validación Cruzada: K-Fold y justificación de Leave-One-Out.

Se comparan el modelo baseline de Regresión Lineal y modelos ensemble ya trabajados en el proyecto.

## 1. Importación de librerías

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. Carga del dataset

El dataset debe estar descargado desde Kaggle y ubicado en `data/raw/train.csv`.

In [2]:
possible_paths = [
    Path("../data/raw/train.csv"),
    Path("data/raw/train.csv"),
    Path("train.csv"),
]

DATA_PATH = next((path for path in possible_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("No se encontro train.csv. Coloca el dataset en data/raw/train.csv.")

df = pd.read_csv(DATA_PATH)
print(f"Dataset cargado desde: {DATA_PATH}")
print(f"Dimensiones: {df.shape}")
df.head()

Dataset cargado desde: ..\data\raw\train.csv
Dimensiones: (1117957, 22)


,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
0,0,5,8,5,8,6,4,4,3,3,...,5,3,3,5,4,7,5,7,3,0.445
1,1,6,7,4,4,8,8,3,5,4,...,7,2,0,3,5,3,3,4,3,0.450
2,2,6,5,6,7,3,7,1,5,4,...,7,3,7,5,6,8,2,3,3,0.530
3,3,3,4,6,5,4,8,4,7,6,...,2,4,7,4,4,6,5,7,5,0.535
4,4,5,3,2,6,4,4,3,3,3,...,2,2,6,6,4,1,2,3,5,0.415


## 3. Preparación de variables

Se elimina `id` porque es un identificador y no una variable predictora. La variable objetivo es `FloodProbability`.

In [3]:
target = "FloodProbability"
feature_cols = [col for col in df.columns if col not in ["id", target]]

X = df[feature_cols]
y = df[target]

print(f"Variables predictoras: {len(feature_cols)}")
print(feature_cols)

Variables predictoras: 20
['MonsoonIntensity', 'TopographyDrainage', 'RiverManagement', 'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality', 'Siltation', 'AgriculturalPractices', 'Encroachments', 'IneffectiveDisasterPreparedness', 'DrainageSystems', 'CoastalVulnerability', 'Landslides', 'Watersheds', 'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss', 'InadequatePlanning', 'PoliticalFactors']


## 4. Muestra de trabajo

El dataset es grande. Para que la validación cruzada sea ejecutable en local, se usa una muestra reproducible.

Si el equipo dispone de más tiempo o recursos, se puede aumentar `SAMPLE_SIZE`.

In [4]:
SAMPLE_SIZE = 100_000

if len(df) > SAMPLE_SIZE:
    sample_df = df.sample(n=SAMPLE_SIZE, random_state=42)
else:
    sample_df = df.copy()

X_sample = sample_df[feature_cols]
y_sample = sample_df[target]

print(f"Muestra usada para validación cruzada: {X_sample.shape}")

Muestra usada para validación cruzada: (100000, 20)


## 5. Configuración de K-Fold

Se usa K-Fold con 5 particiones, mezcla aleatoria y `random_state=42` para asegurar reproducibilidad.

In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

## 6. Modelos a evaluar

Se evalúa el baseline y modelos ensemble. La Regresión Lineal se incluye con escalado para mantener un pipeline reproducible.

In [6]:
models = {
    "Linear Regression (Baseline)": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=80,
        max_depth=12,
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
    ),
}

## 7. Ejecución de validación cruzada

In [7]:
cv_results = []

for model_name, model in models.items():
    print(f"Evaluando: {model_name}")
    scores = cross_validate(
        model,
        X_sample,
        y_sample,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    cv_results.append(
        {
            "modelo": model_name,
            "RMSE_mean": -scores["test_rmse"].mean(),
            "RMSE_std": scores["test_rmse"].std(),
            "MAE_mean": -scores["test_mae"].mean(),
            "MAE_std": scores["test_mae"].std(),
            "R2_mean": scores["test_r2"].mean(),
            "R2_std": scores["test_r2"].std(),
        }
    )

results_cv = pd.DataFrame(cv_results).sort_values("RMSE_mean")
results_cv

Evaluando: Linear Regression (Baseline)
Evaluando: Random Forest
Evaluando: Gradient Boosting


,modelo,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std
0,Linear Regression (Baseline),0.020058,0.000104,0.015804,0.000076,0.845304,0.002237
2,Gradient Boosting,0.027025,0.000099,0.022193,0.000105,0.719201,0.002010
1,Random Forest,0.035728,0.000129,0.029242,0.000128,0.509234,0.002558


## 8. Interpretación de resultados

La tabla anterior permite comparar el rendimiento medio de cada modelo en 5 particiones distintas. También muestra la desviación estándar, que ayuda a evaluar la estabilidad del modelo.

- Menor `RMSE_mean` indica menor error promedio.
- Menor `MAE_mean` indica menor error absoluto promedio.
- Mayor `R2_mean` indica mejor capacidad explicativa.
- Menor desviación estándar indica comportamiento más estable entre folds.

## 9. Justificacion de Leave-One-Out

Leave-One-Out Cross Validation no se ejecuta en este proyecto porque el dataset contiene un volumen muy alto de registros. Esta técnica entrenaría un modelo por cada observación del dataset, lo que haría el coste computacional excesivo para el entorno local del equipo.

Por este motivo, se utiliza K-Fold con 5 particiones como alternativa más equilibrada entre rigor de evaluación y coste computacional.

## 10. Conclusiones

Tras aplicar K-Fold Cross Validation con 5 particiones sobre una muestra reproducible de 100.000 registros, se observa que el modelo con mejor rendimiento medio sigue siendo `Linear Regression (Baseline)`.

Resultados principales:

- Mejor modelo según RMSE medio: `Linear Regression (Baseline)`, con RMSE medio de aproximadamente `0.0201`.
- Mejor modelo según R2 medio: `Linear Regression (Baseline)`, con R2 medio de aproximadamente `0.8453`.
- `Gradient Boosting` obtiene un rendimiento inferior al baseline, con R2 medio aproximado de `0.7192`.
- `Random Forest` obtiene el peor rendimiento de los tres modelos evaluados, con R2 medio aproximado de `0.5092`.
- Las desviaciónes estándar son bajas en los tres modelos, lo que indica estabilidad entre folds.

La validación cruzada confirma que el baseline lineal no depende de una única partición train/test, sino que mantiene un rendimiento estable en diferentes particiones del dataset.

Leave-One-Out Cross Validation no se ejecuta porque el dataset tiene más de un millón de registros. Esta técnica requeriría entrenar un modelo por cada observación, lo que supone un coste computacional inviable para el entorno local del equipo.

Por tanto, para la siguiente fase de optimización de hiperparámetros, se recomienda priorizar `Gradient Boosting` como modelo ensemble candidato, aunque el baseline lineal sigue siendo el modelo con mejor rendimiento global hasta este punto.